In [29]:
from PIL import Image
import torch
import numpy as np
from IPython.display import clear_output
from typing import List
from tqdm import tqdm
import PIL.Image

class RePaint:
    TARGET_SIZE = (512, 512)
    def __init__(self, pipe):
        self.device = pipe.device
        self.pipe = pipe
        self.scheduler = pipe.scheduler

        self.timesteps = pipe.scheduler.timesteps
        self.betas = pipe.scheduler.betas

        self.alphas = 1 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.roll(self.alphas_cumprod, 1)
        self.alphas_cumprod_prev[0] = 1.0
        
        self.sqrt_one_over_alphas = torch.sqrt(1.0 / self.alphas)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)

    @torch.no_grad()
    def __embed_test(self,prompt):
        text_inputs = self.pipe.tokenizer(prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=pipe.tokenizer.model_max_length)
        text_embeddings = self.pipe.text_encoder(text_inputs.input_ids.to(self.device))[0]
        return text_embeddings
    
    @torch.no_grad()
    def __image_to_tensor(self, image: Image.Image) -> torch.Tensor:
        image = image.convert("RGB")
        image = image.resize(RePaint.TARGET_SIZE)

        img_tensor = torch.from_numpy(np.array(image)).float() / 127.5 - 1.0
        img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0).to(self.device, dtype=torch.float16)
    
        init_latents = self.pipe.vae.encode(img_tensor).latent_dist.sample()
        init_latents = init_latents * self.pipe.vae.config.scaling_factor

        return init_latents
    
    @torch.no_grad()
    def __mask_to_tensor(self, image: Image.Image,image_shape) -> torch.Tensor:
        image = image.convert("L")
        latent_h, latent_w = image_shape[2], image_shape[3]
        mask_resized = image.resize((latent_w, latent_h), PIL.Image.NEAREST)
    
        mask_tensor = torch.from_numpy(np.array(mask_resized)).float() / 255.0
        mask_tensor = mask_tensor.unsqueeze(0).unsqueeze(0).to(self.device, dtype=torch.float16)

        return mask_tensor

    @torch.no_grad()
    def __get_jumps(self,jumps_every:int=100, r:int=5) -> List[int]:
        jumps = []
        for i in range(0, torch.max(self.scheduler.timesteps), jumps_every):
            jumps.extend([i] * r)
        jumps.reverse()
        return jumps
    
    @torch.no_grad()
    def __single_reverse_step(self , x: torch.Tensor, t: int, text_embeddings) -> torch.Tensor:
        mean = self.sqrt_one_over_alphas[t] * (x - self.betas[t] * self.pipe.unet(x, t, encoder_hidden_states=text_embeddings).sample / self.sqrt_one_minus_alphas_cumprod[t])
        if t == 0:
            return mean
        else:
            noise = torch.randn_like(x) * torch.sqrt(self.posterior_variance[t])
            return mean + noise
        
    @torch.no_grad()
    def __zero_to_t(self,x_0: torch.Tensor, t: int) -> torch.Tensor:
        if t == 0:
            return x_0
        else:
            return torch.sqrt(self.alphas_cumprod[t]) * x_0 + \
                    torch.sqrt(1.0 - self.alphas_cumprod[t]) * torch.randn_like(x_0)

    def __tensor_to_image(self, tensor):
        final_latents = 1 / self.pipe.vae.config.scaling_factor * tensor
        image = self.pipe.vae.decode(final_latents).sample
            
        # Convert tensor to PIL Image
        image = (image / 2 + 0.5).clamp(0, 1)
        image = image.cpu().permute(0, 2, 3, 1).detach().numpy()[0]
        return PIL.Image.fromarray((image * 255).astype(np.uint8))

    def impaint(self, image: Image.Image, mask: Image.Image, prompt="", j:int=1, r:int = 1, timestamps=-1,jumps_every=50 ):
        jumps = self.__get_jumps(jumps_every=jumps_every,r=r)
        original_tensor = self.__image_to_tensor(image)
        mask_tensor = self.__mask_to_tensor(mask,original_tensor.shape)
        text_embeddings= self.__embed_test(prompt)

        sample = torch.randn_like(original_tensor).to(self.device)
        
        print("beginning impainting")
        for t in tqdm(self.scheduler.timesteps):
            while len(jumps) > 0 & jumps[0] == t:
                jumps = jumps[1:]
                sample = self.__single_reverse_step(sample, t, text_embeddings)

                for override_t in range(t + j, t, -1):
                    sample = self.__single_reverse_step(sample, override_t,text_embeddings)


            x_known = self.__zero_to_t(original_tensor, t)
            x_unknown = self.__single_reverse_step(sample, t, text_embeddings)
            sample = mask_tensor * x_known + (1-mask_tensor) * x_unknown

        return self.__tensor_to_image(sample)


In [4]:
from diffusers import StableDiffusionPipeline, DDPMScheduler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "sd2-community/stable-diffusion-2-base"
pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16
    ).to(device)

pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

In [5]:
import PIL.Image
source_img = PIL.Image.open("a.png")
# source_img.show()
mask_img = PIL.Image.open("a_mask.png")
# mask_img.show()

In [30]:
r = RePaint(pipe)
print(r)

In [31]:
r.impaint(source_img,mask_img,"christmas tree")

beginning impainting


100%|█████████▉| 999/1000 [01:10<00:00, 14.26it/s]


IndexError: list index out of range

In [9]:
pipe.scheduler.timesteps

tensor([999, 998, 997, 996, 995, 994, 993, 992, 991, 990, 989, 988, 987, 986,
        985, 984, 983, 982, 981, 980, 979, 978, 977, 976, 975, 974, 973, 972,
        971, 970, 969, 968, 967, 966, 965, 964, 963, 962, 961, 960, 959, 958,
        957, 956, 955, 954, 953, 952, 951, 950, 949, 948, 947, 946, 945, 944,
        943, 942, 941, 940, 939, 938, 937, 936, 935, 934, 933, 932, 931, 930,
        929, 928, 927, 926, 925, 924, 923, 922, 921, 920, 919, 918, 917, 916,
        915, 914, 913, 912, 911, 910, 909, 908, 907, 906, 905, 904, 903, 902,
        901, 900, 899, 898, 897, 896, 895, 894, 893, 892, 891, 890, 889, 888,
        887, 886, 885, 884, 883, 882, 881, 880, 879, 878, 877, 876, 875, 874,
        873, 872, 871, 870, 869, 868, 867, 866, 865, 864, 863, 862, 861, 860,
        859, 858, 857, 856, 855, 854, 853, 852, 851, 850, 849, 848, 847, 846,
        845, 844, 843, 842, 841, 840, 839, 838, 837, 836, 835, 834, 833, 832,
        831, 830, 829, 828, 827, 826, 825, 824, 823, 822, 821, 8